<a href="https://colab.research.google.com/github/Sprg72/Data-Engineer/blob/main/notebooks/Pyspark_dataframes2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# spark dataframes.
# ---> provides structure (tabular) shape to data.
# difference between Pandas dataframes and Spark Dataframes
# pandas dataframes not distributed objects --> no parallel process.
# spark dataframes are distributed objects --> MPP with In memory computing .


In [1]:
# step1
!pip install findspark pyspark

In [2]:
# step2
import findspark
findspark.init()

In [3]:
#step3
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('myapp').getOrCreate()

In [4]:
sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=myapp>

In [5]:
# create a dataframe from local object.
# newdf = spark.createDataFrame(input)  # input should be list of tuples.

data = [(101, 'Ravi'), (102, 'Rani'), (103, 'Venu')]
df = spark.createDataFrame(data)
df.show()


+---+----+
| _1|  _2|
+---+----+
|101|Ravi|
|102|Rani|
|103|Venu|
+---+----+





```
# This is formatted as code

input file : emp.txt

id,name,salary,gender,dno
101,amar,90000,m,11
102,amala,20000,f,12
103,ankit,40000,m,13
104,ankita,60000,f,13
105,anusha,110000,f,12
106,anuz,20000,m,11
107,akash,100000,m,12
```



In [6]:
df = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/emp.txt")

df.show()


+---+------+------+------+---+
| id|  name|salary|gender|dno|
+---+------+------+------+---+
|101|  amar| 90000|     m| 11|
|102| amala| 20000|     f| 12|
|103| ankit| 40000|     m| 13|
|104|ankita| 60000|     f| 13|
|105|anusha|110000|     f| 12|
|106|  anuz| 20000|     m| 11|
|107| akash|100000|     m| 12|
+---+------+------+------+---+



In [7]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- dno: integer (nullable = true)





```
# This is formatted as code

input file: emp1.txt

101, amar,90000,m,11
102, amala,20000,f,12
103, ankit,40000,m,13
104, ankita,60000,f,13
105, anusha,110000,f,12
106, anuz,20000,m,11
107, akash,100000,m,12
108, siva,20000,m,14
109, sivani,30000,f,15
110, mani,30000,m,12
111, manisha,300000,f,13
112, sivam,200000,m,12
113, varun,200000,m,13

```



In [8]:
# case2: file does not have header
# create custom schema.
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

myschema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("gender", StringType(), True),
    StructField("dno", IntegerType(), True)
])


In [32]:
df = spark.read.option("header", "false") \
    .schema(myschema) \
    .csv("/content/emp1.txt")

df.show()

+---+--------+--------+------+---+
| id|    name|  salary|gender|dno|
+---+--------+--------+------+---+
|101|    amar| 90000.0|     m| 11|
|102|   amala| 20000.0|     f| 12|
|103|   ankit| 40000.0|     m| 13|
|104|  ankita| 60000.0|     f| 13|
|105|  anusha|110000.0|     f| 12|
|106|    anuz| 20000.0|     m| 11|
|107|   akash|100000.0|     m| 12|
|108|    siva| 20000.0|     m| 14|
|109|  sivani| 30000.0|     f| 15|
|110|    mani| 30000.0|     m| 12|
|111| manisha|300000.0|     f| 13|
|112|   sivam|200000.0|     m| 12|
|113|   varun|200000.0|     m| 13|
+---+--------+--------+------+---+



In [12]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- dno: integer (nullable = true)





```
# This is formatted as code

input file : prods.txt

p1	tv	500000	Samsung
p2	lap	1000000	Lg
p3	iPhone	3000000	Apple
```



In [13]:
#case3: comma is not delimiter, tab space is delimiter
prodschema = StructType([
    StructField("pid", StringType(), False),
    StructField("pname", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("brand", StringType(), True)
])

products = spark.read.option("header", "false") \
    .option("sep", "\t") \
    .schema(prodschema) \
    .csv('/content/prods.txt')

products.show()


+---+------+-------+-------+
|pid| pname|  price|  brand|
+---+------+-------+-------+
| p1|    tv| 500000|Samsung|
| p2|   lap|1000000|     Lg|
| p3|iPhone|3000000|  Apple|
+---+------+-------+-------+



In [14]:
products.printSchema()

root
 |-- pid: string (nullable = true)
 |-- pname: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- brand: string (nullable = true)



In [33]:
# transformations
# filter transformations
# fetch only males data.
from pyspark.sql.functions import col
males = df.filter(col("gender") == 'm')
males.show()

+---+------+--------+------+---+
| id|  name|  salary|gender|dno|
+---+------+--------+------+---+
|101|  amar| 90000.0|     m| 11|
|103| ankit| 40000.0|     m| 13|
|106|  anuz| 20000.0|     m| 11|
|107| akash|100000.0|     m| 12|
|108|  siva| 20000.0|     m| 14|
|110|  mani| 30000.0|     m| 12|
|112| sivam|200000.0|     m| 12|
|113| varun|200000.0|     m| 13|
+---+------+--------+------+---+



In [34]:
# multiple conditions
# males from 11 dno.

males11 = df.filter((col("gender") == 'm') & (col("dno") == 11))
males11.show()

+---+-----+-------+------+---+
| id| name| salary|gender|dno|
+---+-----+-------+------+---+
|101| amar|90000.0|     m| 11|
|106| anuz|20000.0|     m| 11|
+---+-----+-------+------+---+



In [35]:
# females from 11, 13 dno's
fem12and13 = df.filter((col("gender") == 'f') & ((col("dno") == 12) | (col("dno") == 13)))
fem12and13.show()

+---+--------+--------+------+---+
| id|    name|  salary|gender|dno|
+---+--------+--------+------+---+
|102|   amala| 20000.0|     f| 12|
|104|  ankita| 60000.0|     f| 13|
|105|  anusha|110000.0|     f| 12|
|111| manisha|300000.0|     f| 13|
+---+--------+--------+------+---+



In [36]:
# row level transformations
# withColumn() is used.
# 1.update 2.create
# if given column existing, it perform "update"
# if given column does not existing, it creates a new column
# df.withColumn("colname", expression)

In [37]:
df.show()

+---+--------+--------+------+---+
| id|    name|  salary|gender|dno|
+---+--------+--------+------+---+
|101|    amar| 90000.0|     m| 11|
|102|   amala| 20000.0|     f| 12|
|103|   ankit| 40000.0|     m| 13|
|104|  ankita| 60000.0|     f| 13|
|105|  anusha|110000.0|     f| 12|
|106|    anuz| 20000.0|     m| 11|
|107|   akash|100000.0|     m| 12|
|108|    siva| 20000.0|     m| 14|
|109|  sivani| 30000.0|     f| 15|
|110|    mani| 30000.0|     m| 12|
|111| manisha|300000.0|     f| 13|
|112|   sivam|200000.0|     m| 12|
|113|   varun|200000.0|     m| 13|
+---+--------+--------+------+---+



In [39]:
from pyspark.sql.functions import round
df = df.withColumn("salary", round(col("salary") * 1.1, 2))
df.show()

+---+--------+--------+------+---+
| id|    name|  salary|gender|dno|
+---+--------+--------+------+---+
|101|    amar|108900.0|     m| 11|
|102|   amala| 24200.0|     f| 12|
|103|   ankit| 48400.0|     m| 13|
|104|  ankita| 72600.0|     f| 13|
|105|  anusha|133100.0|     f| 12|
|106|    anuz| 24200.0|     m| 11|
|107|   akash|121000.0|     m| 12|
|108|    siva| 24200.0|     m| 14|
|109|  sivani| 36300.0|     f| 15|
|110|    mani| 36300.0|     m| 12|
|111| manisha|363000.0|     f| 13|
|112|   sivam|242000.0|     m| 12|
|113|   varun|242000.0|     m| 13|
+---+--------+--------+------+---+



In [40]:
df = df.withColumn("tax", round(col("salary") * 0.1, 2))
df.show()

+---+--------+--------+------+---+-------+
| id|    name|  salary|gender|dno|    tax|
+---+--------+--------+------+---+-------+
|101|    amar|108900.0|     m| 11|10890.0|
|102|   amala| 24200.0|     f| 12| 2420.0|
|103|   ankit| 48400.0|     m| 13| 4840.0|
|104|  ankita| 72600.0|     f| 13| 7260.0|
|105|  anusha|133100.0|     f| 12|13310.0|
|106|    anuz| 24200.0|     m| 11| 2420.0|
|107|   akash|121000.0|     m| 12|12100.0|
|108|    siva| 24200.0|     m| 14| 2420.0|
|109|  sivani| 36300.0|     f| 15| 3630.0|
|110|    mani| 36300.0|     m| 12| 3630.0|
|111| manisha|363000.0|     f| 13|36300.0|
|112|   sivam|242000.0|     m| 12|24200.0|
|113|   varun|242000.0|     m| 13|24200.0|
+---+--------+--------+------+---+-------+



In [42]:
df = df.withColumn("hra", col("salary")*0.2) \
       .withColumn("netsal", col("salary") - col("tax") + col("hra"))
df.show()

+---+--------+--------+------+---+-------+-------+--------+
| id|    name|  salary|gender|dno|    tax|    hra|  netsal|
+---+--------+--------+------+---+-------+-------+--------+
|101|    amar|108900.0|     m| 11|10890.0|21780.0|119790.0|
|102|   amala| 24200.0|     f| 12| 2420.0| 4840.0| 26620.0|
|103|   ankit| 48400.0|     m| 13| 4840.0| 9680.0| 53240.0|
|104|  ankita| 72600.0|     f| 13| 7260.0|14520.0| 79860.0|
|105|  anusha|133100.0|     f| 12|13310.0|26620.0|146410.0|
|106|    anuz| 24200.0|     m| 11| 2420.0| 4840.0| 26620.0|
|107|   akash|121000.0|     m| 12|12100.0|24200.0|133100.0|
|108|    siva| 24200.0|     m| 14| 2420.0| 4840.0| 26620.0|
|109|  sivani| 36300.0|     f| 15| 3630.0| 7260.0| 39930.0|
|110|    mani| 36300.0|     m| 12| 3630.0| 7260.0| 39930.0|
|111| manisha|363000.0|     f| 13|36300.0|72600.0|399300.0|
|112|   sivam|242000.0|     m| 12|24200.0|48400.0|266200.0|
|113|   varun|242000.0|     m| 13|24200.0|48400.0|266200.0|
+---+--------+--------+------+---+------

In [44]:
# df.select() ---> to fetch specific columns, or to reorder columns.
df.select("id", "name", "salary", "netsal").show(3)

+---+------+--------+--------+
| id|  name|  salary|  netsal|
+---+------+--------+--------+
|101|  amar|108900.0|119790.0|
|102| amala| 24200.0| 26620.0|
|103| ankit| 48400.0| 53240.0|
+---+------+--------+--------+
only showing top 3 rows


In [46]:
df = df.select("id", "name", "salary", "tax", "hra", "netsal", "gender", "dno")
df.show()

+---+--------+--------+-------+-------+--------+------+---+
| id|    name|  salary|    tax|    hra|  netsal|gender|dno|
+---+--------+--------+-------+-------+--------+------+---+
|101|    amar|108900.0|10890.0|21780.0|119790.0|     m| 11|
|102|   amala| 24200.0| 2420.0| 4840.0| 26620.0|     f| 12|
|103|   ankit| 48400.0| 4840.0| 9680.0| 53240.0|     m| 13|
|104|  ankita| 72600.0| 7260.0|14520.0| 79860.0|     f| 13|
|105|  anusha|133100.0|13310.0|26620.0|146410.0|     f| 12|
|106|    anuz| 24200.0| 2420.0| 4840.0| 26620.0|     m| 11|
|107|   akash|121000.0|12100.0|24200.0|133100.0|     m| 12|
|108|    siva| 24200.0| 2420.0| 4840.0| 26620.0|     m| 14|
|109|  sivani| 36300.0| 3630.0| 7260.0| 39930.0|     f| 15|
|110|    mani| 36300.0| 3630.0| 7260.0| 39930.0|     m| 12|
|111| manisha|363000.0|36300.0|72600.0|399300.0|     f| 13|
|112|   sivam|242000.0|24200.0|48400.0|266200.0|     m| 12|
|113|   varun|242000.0|24200.0|48400.0|266200.0|     m| 13|
+---+--------+--------+-------+-------+-

In [47]:
df.write.option("header", "true") \
    .mode("overwrite") \
    .csv("/content/emptransformed")



```
# This is formatted as code
output file: part-00000
id,name,salary,tax,hra,netsal,gender,dno
101,amar,108900.0,10890.0,21780.0,119790.0,m,11
102,amala,24200.0,2420.0,4840.0,26620.0,f,12
103,ankit,48400.0,4840.0,9680.0,53240.0,m,13
104,ankita,72600.0,7260.0,14520.0,79860.0,f,13
105,anusha,133100.0,13310.0,26620.0,146410.0,f,12
106,anuz,24200.0,2420.0,4840.0,26620.0,m,11
107,akash,121000.0,12100.0,24200.0,133100.0,m,12
108,siva,24200.0,2420.0,4840.0,26620.0,m,14
109,sivani,36300.0,3630.0,7260.0,39930.0,f,15
110,mani,36300.0,3630.0,7260.0,39930.0,m,12
111,manisha,363000.0,36300.0,72600.0,399300.0,f,13
112,sivam,242000.0,24200.0,48400.0,266200.0,m,12
113,varun,242000.0,24200.0,48400.0,266200.0,m,13

```



In [48]:
# conditional transformations
# "when" function is used to perform conditional transformatios.
from pyspark.sql.functions import when

In [49]:
df = df.withColumn(
    "gender",
    when(col("gender") == 'm', "Male")
        .when(col("gender") == 'f', "Female")
        .otherwise("Invalid")
)

df.show()


+---+--------+--------+-------+-------+--------+------+---+
| id|    name|  salary|    tax|    hra|  netsal|gender|dno|
+---+--------+--------+-------+-------+--------+------+---+
|101|    amar|108900.0|10890.0|21780.0|119790.0|  Male| 11|
|102|   amala| 24200.0| 2420.0| 4840.0| 26620.0|Female| 12|
|103|   ankit| 48400.0| 4840.0| 9680.0| 53240.0|  Male| 13|
|104|  ankita| 72600.0| 7260.0|14520.0| 79860.0|Female| 13|
|105|  anusha|133100.0|13310.0|26620.0|146410.0|Female| 12|
|106|    anuz| 24200.0| 2420.0| 4840.0| 26620.0|  Male| 11|
|107|   akash|121000.0|12100.0|24200.0|133100.0|  Male| 12|
|108|    siva| 24200.0| 2420.0| 4840.0| 26620.0|  Male| 14|
|109|  sivani| 36300.0| 3630.0| 7260.0| 39930.0|Female| 15|
|110|    mani| 36300.0| 3630.0| 7260.0| 39930.0|  Male| 12|
|111| manisha|363000.0|36300.0|72600.0|399300.0|Female| 13|
|112|   sivam|242000.0|24200.0|48400.0|266200.0|  Male| 12|
|113|   varun|242000.0|24200.0|48400.0|266200.0|  Male| 13|
+---+--------+--------+-------+-------+-

In [50]:
df = df.withColumn(
    "dname",
    when(col('dno') == 11, 'Marketing')
        .when(col('dno') == 12, 'Hr')
        .when(col('dno') == 13, 'Finance')
        .otherwise("Other")
)

df.show()


+---+--------+--------+-------+-------+--------+------+---+---------+
| id|    name|  salary|    tax|    hra|  netsal|gender|dno|    dname|
+---+--------+--------+-------+-------+--------+------+---+---------+
|101|    amar|108900.0|10890.0|21780.0|119790.0|  Male| 11|Marketing|
|102|   amala| 24200.0| 2420.0| 4840.0| 26620.0|Female| 12|       Hr|
|103|   ankit| 48400.0| 4840.0| 9680.0| 53240.0|  Male| 13|  Finance|
|104|  ankita| 72600.0| 7260.0|14520.0| 79860.0|Female| 13|  Finance|
|105|  anusha|133100.0|13310.0|26620.0|146410.0|Female| 12|       Hr|
|106|    anuz| 24200.0| 2420.0| 4840.0| 26620.0|  Male| 11|Marketing|
|107|   akash|121000.0|12100.0|24200.0|133100.0|  Male| 12|       Hr|
|108|    siva| 24200.0| 2420.0| 4840.0| 26620.0|  Male| 14|    Other|
|109|  sivani| 36300.0| 3630.0| 7260.0| 39930.0|Female| 15|    Other|
|110|    mani| 36300.0| 3630.0| 7260.0| 39930.0|  Male| 12|       Hr|
|111| manisha|363000.0|36300.0|72600.0|399300.0|Female| 13|  Finance|
|112|   sivam|242000

In [51]:
df = df.withColumn(
    "salary_category",
    when(col('salary') >= 200000, 'High')
        .when(col('salary') >= 100000, 'Medium')
        .otherwise("Low")
)

df.show()


+---+--------+--------+-------+-------+--------+------+---+---------+---------------+
| id|    name|  salary|    tax|    hra|  netsal|gender|dno|    dname|salary_category|
+---+--------+--------+-------+-------+--------+------+---+---------+---------------+
|101|    amar|108900.0|10890.0|21780.0|119790.0|  Male| 11|Marketing|         Medium|
|102|   amala| 24200.0| 2420.0| 4840.0| 26620.0|Female| 12|       Hr|            Low|
|103|   ankit| 48400.0| 4840.0| 9680.0| 53240.0|  Male| 13|  Finance|            Low|
|104|  ankita| 72600.0| 7260.0|14520.0| 79860.0|Female| 13|  Finance|            Low|
|105|  anusha|133100.0|13310.0|26620.0|146410.0|Female| 12|       Hr|         Medium|
|106|    anuz| 24200.0| 2420.0| 4840.0| 26620.0|  Male| 11|Marketing|            Low|
|107|   akash|121000.0|12100.0|24200.0|133100.0|  Male| 12|       Hr|         Medium|
|108|    siva| 24200.0| 2420.0| 4840.0| 26620.0|  Male| 14|    Other|            Low|
|109|  sivani| 36300.0| 3630.0| 7260.0| 39930.0|Female